## Collapse VASA data onto lineage tree structure

In [ ]:
import sys
import random
import warnings
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", UserWarning)

import dill
import random

random.seed(10)

from anndata import AnnData
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn
seaborn.reset_orig()
%matplotlib inline

sc.set_figure_params(figsize=(10, 10))

## Load data

In [ ]:
out_file = "/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa.processed.eecs.output.scfates.h5ad"
adata = sc.read(out_file)

In [ ]:
sc.pl.umap(adata, color=["LMX1B", "CHGA"], cmap="RdPu", frameon=False)

In [ ]:
sc.pl.umap(adata, color=["LMX1B", "KCNB2"], cmap="RdPu", frameon=False)

In [ ]:
sc.pl.umap(adata, color=["LMX1B", "FEV"], cmap="RdPu", frameon=False)

In [ ]:
sc.pl.umap(adata, color=["LMX1B", "PAX4"], cmap="RdPu", frameon=False)

## Collapse onto milestones / tree structure

In [ ]:
import random
import numpy as np
import pandas as pd
import igraph
import networkx as nx
import matplotlib.pyplot as plt

# Tree layout helper
def hierarchy_pos(G, root=None, width=1.0, vert_gap=0.2, vert_loc=0, xcenter=0.5):
    if not nx.is_tree(G):
        raise TypeError("hierarchy_pos can only be used on a tree.")

    # Pick root if not given
    if root is None:
        root = next(iter(nx.topological_sort(G))) if isinstance(G, nx.DiGraph) else random.choice(list(G.nodes))

    def recurse(node, width, vert_loc, xcenter, pos, parent=None):
        pos[node] = (xcenter, vert_loc)
        children = list(G.neighbors(node))

        # When graph is undirected, exclude parent in recursion
        if parent is not None and not isinstance(G, nx.DiGraph):
            children = [c for c in children if c != parent]

        if children:
            dx = width / len(children)
            x = xcenter - width / 2 - dx / 2
            for child in children:
                x += dx
                recurse(child, width=dx, vert_loc=vert_loc - vert_gap, xcenter=x, pos=pos, parent=node)

        return pos

    return recurse(root, width, vert_loc, xcenter, pos={})


def build_cell_lineage_graph(graph):
    milestone_dict = graph["milestones"]

    # Convert to strings for compatibility
    milestone_ids = [str(v) for v in milestone_dict.values()]
    milestone_labels = list(milestone_dict.keys())

    # Build igraph
    g = igraph.Graph(directed=True)
    g.add_vertices(len(milestone_ids))

    # Assign vertex names and labels
    g.vs["name"] = milestone_ids
    g.vs["label"] = milestone_labels

    # Build edges
    edge_tuples = graph["pp_seg"][["from", "to"]].astype(str).apply(tuple, axis=1).tolist()
    g.add_edges(edge_tuples)

    # Convert to networkx
    G = g.to_networkx()

    # Build dict to rename nodes based on milestone labels
    id_to_label = {i: milestone_labels[i] for i in range(len(milestone_ids))}

    # Relabel nodes in NetworkX
    G = nx.relabel_nodes(G, id_to_label)

    return G

In [ ]:
# Build lineage graph and apply milestone name mapping
G_raw = build_cell_lineage_graph(adata.uns["graph"])

# Milestone ID → human-readable milestone name
node_mapping = {
    "132": "X cells",
    "137": "D cells",
    "187": "D/I/K cells",
    "219": "EC/X/D/I/K cells",
    "228": "X/D/I/K cells",
    "232": "NGN3+ cells",
    "244": "EC cells",
    "269": "I/K cells",
}

# Relabel graph nodes
G = nx.relabel_nodes(G_raw, node_mapping)

# Compute hierarchical layout and remap its keys
pos_raw = hierarchy_pos(G_raw)
pos = {node_mapping[k]: v for k, v in pos_raw.items()}

# Compute mean expression per milestone

# Dense gene expression table with milestone assignments
df = sc.get.obs_df(adata, list(adata.var_names))
df["milestones"] = adata.obs["milestones"].map(node_mapping)

# Average expression for each milestone node
df_mean = df.groupby("milestones").mean()


# Sort nodes based on hierarchical layout order

# Sort nodes by their vertical position (y-coordinate)
milestone_order = sorted(pos.keys(), key=lambda n: pos[n][1], reverse=True)

# Reorder layout and expression accordingly
pos = {m: pos[m] for m in milestone_order}
df_mean = df_mean.reindex(milestone_order)

# Rebuild graph to enforce milestone order
G_new = nx.DiGraph()
G_new.add_nodes_from(milestone_order)
G_new.add_edges_from(G.edges())   # edges preserve parent–child relationships

lineage_model = {
    "graph": G_new,
    "positions": pos,
    "mean_expression": df_mean,
    "node_order": milestone_order,
}

# Save lineage model as a pickle file using dill
with open("/projects/site/pred/ihb-intestine-evo/lukas_area/eec_fate_project/processed/vasa_lineage_model.pkl", "wb") as f:
    dill.dump(lineage_model, f)

In [ ]:
gene = "PAX4"
vals = lineage_model["mean_expression"][gene]

fig, ax = plt.subplots(figsize=(6, 6))

nodes = nx.draw(
    lineage_model["graph"],
    pos=lineage_model["positions"],
    node_color=vals.values,
    node_size=800,
    cmap="viridis",
    with_labels=True,
    font_size=8,
    ax=ax
)

# --- Add colorbar legend ---
sm = plt.cm.ScalarMappable(cmap="viridis", 
                           norm=plt.Normalize(vmin=vals.min(), vmax=vals.max()))
sm.set_array([])  # required in newer Matplotlib versions
cbar = plt.colorbar(sm, ax=ax)
cbar.set_label(f"Average expression of {gene}")

plt.show()